In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.transforms.functional as TF



if 'path' not in locals() and 'path' not in globals():
    print("\"path\" variable not found, attempting to re-download dataset...")
    path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")
    print("Path to dataset files (defined in this cell):", path)

# 1) Auto-detect image/mask folders

def find_first_folder_with_images(root, exts=("jpg","jpeg","png")):
    for p in sorted(glob.glob(os.path.join(root, "**"), recursive=True)):
        if os.path.isdir(p):
            files = []
            for e in exts:
                files += glob.glob(os.path.join(p, f"*.{e}"))
            if len(files) > 20:
                return p
    return None

def guess_image_mask_dirs(dataset_root):

    candidates_img = ["images", "Imgs", "img", "image", "Images", "RGB", "rgb"]
    candidates_msk = ["masks", "Masks", "mask", "labels", "Labels", "GT", "gt", "Seg", "seg"]

    img_dir = None
    msk_dir = None


    children = [os.path.join(dataset_root, d) for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))]
    lower_map = {os.path.basename(c).lower(): c for c in children}

    for c in candidates_img:
        if c.lower() in lower_map:
            img_dir = lower_map[c.lower()]
            break
    for c in candidates_msk:
        if c.lower() in lower_map:
            msk_dir = lower_map[c.lower()]
            break


    if img_dir is None:
        img_dir = find_first_folder_with_images(dataset_root, exts=("jpg","jpeg","png"))
    if msk_dir is None:
        msk_dir = find_first_folder_with_images(dataset_root, exts=("png",))

    return img_dir, msk_dir

img_dir, mask_dir = guess_image_mask_dirs(path)
print("Detected img_dir :", img_dir)
print("Detected mask_dir:", mask_dir)

assert img_dir is not None and mask_dir is not None, "Could not detect images/masks folders. Print `path` and check the dataset structure."


In [ ]:
# 2) Build paired file lists

def build_pairs(img_dir, mask_dir):
    img_files = []
    for ext in ("*.jpg","*.jpeg","*.png"):
        img_files += glob.glob(os.path.join(img_dir, ext))
    mask_files = glob.glob(os.path.join(mask_dir, "*.png"))

    img_map  = {os.path.splitext(os.path.basename(f))[0]: f for f in img_files}
    mask_map = {os.path.splitext(os.path.basename(f))[0]: f for f in mask_files}

    common = sorted(set(img_map.keys()) & set(mask_map.keys()))
    assert len(common) > 0, "No matching image/mask stems found. Check naming."

    pairs = [(img_map[k], mask_map[k]) for k in common]
    return pairs

pairs = build_pairs(img_dir, mask_dir)
print("Total paired samples:", len(pairs))


In [ ]:

class SUIMDataset(Dataset):

    def __init__(self, pairs, img_size=(256, 256), augment=False):
        self.pairs = pairs
        self.img_size = img_size
        self.augment = augment

        self.img_tf = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std =[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        img  = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # Resize: image bilinear, mask nearest
        img  = TF.resize(img,  self.img_size, interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.img_size, interpolation=TF.InterpolationMode.NEAREST)


        if self.augment:
            if torch.rand(1).item() < 0.5:
                img  = TF.hflip(img)
                mask = TF.hflip(mask)
            if torch.rand(1).item() < 0.25:
                img  = TF.vflip(img)
                mask = TF.vflip(mask)

        img_t = self.img_tf(img)


        mask_np = np.array(mask)
        mask_t = torch.from_numpy(mask_np).long()

        mask_t = remap_mask(mask_t)

        return img_t, mask_t


# 4) Split + Dataloaders
full_ds = SUIMDataset(pairs, img_size=(256,256), augment=True)

train_size = int(0.8 * len(full_ds))
val_size   = len(full_ds) - train_size

train_ds, val_ds = random_split(full_ds, [train_size, val_size], generator=torch.Generator().manual_seed(42))

val_ds.dataset.augment = False

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

# 5) Visualize some samples
def show_samples(loader, n=3):
    imgs, masks = next(iter(loader))
    n = min(n, imgs.size(0))
    plt.figure(figsize=(10, 3*n))
    for i in range(n):
        # denormalize for display
        img = imgs[i].clone()
        img = img * torch.tensor([0.229,0.224,0.225]).view(3,1,1) + torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        img = img.clamp(0,1).permute(1,2,0).numpy()

        plt.subplot(n, 2, 2*i+1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(n, 2, 2*i+2)
        plt.imshow(masks[i].numpy(), vmin=0, vmax=7)
        plt.title("Mask (class IDs)")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_loader, n=3)

In [ ]:
# TO DO

try:
    import segmentation_models_pytorch as smp
except Exception:
    !pip -q install segmentation-models-pytorch
    import segmentation_models_pytorch as smp

import torch

NUM_CLASSES = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None  # logits for CrossEntropyLoss
).to(device)

print(model.__class__.__name__)

from tqdm import tqdm
import torch.nn as nn

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    total_pixels = 0

    for imgs, masks in tqdm(loader, desc="Train", leave=False):
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(imgs)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        total_pixels += imgs.size(0)

    return running_loss / max(total_pixels, 1)


In [ ]:
# TO DO
from tqdm import tqdm
import torch.nn as nn

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    total_pixels = 0

    for imgs, masks in tqdm(loader, desc="Train", leave=False):
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(imgs)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        total_pixels += imgs.size(0)

    return running_loss / max(total_pixels, 1)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total_pixels = 0

    for imgs, masks in tqdm(loader, desc="Val", leave=False):
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(imgs)
        loss = criterion(logits, masks)

        running_loss += loss.item() * imgs.size(0)
        total_pixels += imgs.size(0)

    return running_loss / max(total_pixels, 1)

In [ ]:
# TO DO
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

criterion = nn.CrossEntropyLoss()   # multi-class segmentation
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 8
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss = validate(model, val_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f}")

# Plot
plt.figure(figsize=(7,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TO DO
import numpy as np
import matplotlib.pyplot as plt
import torch

@torch.no_grad()
def visualize_predictions(model, loader, device, n=3):
    model.eval()
    imgs, masks = next(iter(loader))
    imgs = imgs.to(device)

    logits = model(imgs)
    preds  = torch.argmax(logits, dim=1).cpu()

    imgs = imgs.cpu()
    masks = masks.cpu()

    n = min(n, imgs.size(0))
    plt.figure(figsize=(12, 4*n))

    for i in range(n):
        # denormalize
        img = imgs[i].clone()
        img = img * torch.tensor([0.229,0.224,0.225]).view(3,1,1) + torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        img = img.clamp(0,1).permute(1,2,0).numpy()

        gt = masks[i].numpy()
        pr = preds[i].numpy()

        plt.subplot(n, 3, 3*i+1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(n, 3, 3*i+2)
        plt.imshow(gt, vmin=0, vmax=7)
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(n, 3, 3*i+3)
        plt.imshow(pr, vmin=0, vmax=7)
        plt.title("Prediction")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

visualize_predictions(model, val_loader, device, n=3)